In [79]:
import pandas as pd

historical_df = pd.read_csv(
    "../data/processed/historical_matches_with_elo.csv"
)

historical_df["date"] = pd.to_datetime(
    historical_df["date"]
)

historical_df["match_id"] = historical_df.index

team_matches_df = pd.read_csv(
    "../data/interim/team_matches.csv"
)

team_matches_df["date"] = pd.to_datetime(
    team_matches_df["date"]
)

In [80]:
historical_df.shape
historical_df.head()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral,home_elo,away_elo,elo_diff,match_id
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False,1500.000000,1500.000000,0.000000,0
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False,1500.000000,1500.000000,0.000000,1
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False,1490.000000,1510.000000,-20.000000,2
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False,1499.424989,1500.575011,-1.150023,3
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False,1500.541911,1499.458089,1.083822,4


In [81]:
home_features = team_matches_df[
    [
        "match_id",
        "team",
        "form_points_5",
        "avg_goals_for_5",
        "avg_goals_against_5",
        "avg_goal_diff_5"
    ]
].copy()

In [82]:
home_features = home_features.rename(
    columns={
        "team": "home_team",
        "form_points_5": "home_form_5",
        "avg_goals_for_5": "home_avg_goals_5",
        "avg_goals_against_5": "home_avg_conceded_5",
        "avg_goal_diff_5": "home_avg_goal_diff_5"
    }
)

In [83]:
home_features = home_features.merge(
    historical_df[
        ["match_id", "home_team"]
    ],
    on=["match_id", "home_team"],
    how="inner"
)

home_features.shape

(23176, 6)

In [84]:
away_features = team_matches_df[
    [
        "match_id",
        "team",
        "form_points_5",
        "avg_goals_for_5",
        "avg_goals_against_5",
        "avg_goal_diff_5"
    ]
].copy()

In [85]:
away_features = away_features.rename(
    columns={
        "team": "away_team",
        "form_points_5": "away_form_5",
        "avg_goals_for_5": "away_avg_goals_5",
        "avg_goals_against_5": "away_avg_conceded_5",
        "avg_goal_diff_5": "away_avg_goal_diff_5"
    }
)

In [86]:
away_features = away_features.merge(
    historical_df[
        ["match_id", "away_team"]
    ],
    on=["match_id", "away_team"],
    how="inner"
)

away_features.shape

(23170, 6)

In [87]:
model_df = historical_df.merge(
    home_features,
    on="match_id",
    how="left"
)

model_df.shape

(49404, 18)

In [88]:
model_df = model_df.merge(
    away_features,
    on="match_id",
    how="left"
)

model_df.shape

(49404, 23)

In [89]:
model_df["goal_difference"] = (
    model_df["home_score"]
    - model_df["away_score"]
)

In [90]:
print("Historical:", historical_df.shape)
print("Model:", model_df.shape)
print(model_df.shape)

model_df.head()

Historical: (49404, 13)
Model: (49404, 24)
(49404, 24)


,date,home_team_x,away_team_x,home_score,away_score,tournament,city,country,neutral,home_elo,...,home_form_5,home_avg_goals_5,home_avg_conceded_5,home_avg_goal_diff_5,away_team_y,away_form_5,away_avg_goals_5,away_avg_conceded_5,away_avg_goal_diff_5,goal_difference
0,1872-11-30,Scotland,England,0,0,Friendly,Glasgow,Scotland,False,1500.000000,...,NaN,NaN,NaN,NaN,England,NaN,NaN,NaN,NaN,0
1,1873-03-08,England,Scotland,4,2,Friendly,London,England,False,1500.000000,...,1.0,0.000000,0.000000,0.000000,Scotland,1.0,0.000000,0.000000,0.000000,2
2,1874-03-07,Scotland,England,2,1,Friendly,Glasgow,Scotland,False,1490.000000,...,1.0,1.000000,2.000000,-1.000000,England,4.0,2.000000,1.000000,1.000000,1
3,1875-03-06,England,Scotland,2,2,Friendly,London,England,False,1499.424989,...,4.0,1.666667,1.333333,0.333333,Scotland,4.0,1.333333,1.666667,-0.333333,0
4,1876-03-04,Scotland,England,3,0,Friendly,Glasgow,Scotland,False,1500.541911,...,5.0,1.500000,1.750000,-0.250000,England,5.0,1.750000,1.500000,0.250000,3


In [91]:
model_df = model_df.drop(
    columns=[
        "home_team_y",
        "away_team_y"
    ]
)

model_df = model_df.rename(
    columns={
        "home_team_x": "home_team",
        "away_team_x": "away_team"
    }
)

model_df = model_df.dropna().reset_index(drop=True)

In [92]:
print(model_df.shape)

(22924, 22)


In [93]:
model_df.shape

(22924, 22)

In [94]:
model_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 22924 entries, 0 to 22923
Data columns (total 22 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   date                  22924 non-null  datetime64[us]
 1   home_team             22924 non-null  str           
 2   away_team             22924 non-null  str           
 3   home_score            22924 non-null  int64         
 4   away_score            22924 non-null  int64         
 5   tournament            22924 non-null  str           
 6   city                  22924 non-null  str           
 7   country               22924 non-null  str           
 8   neutral               22924 non-null  bool          
 9   home_elo              22924 non-null  float64       
 10  away_elo              22924 non-null  float64       
 11  elo_diff              22924 non-null  float64       
 12  match_id              22924 non-null  int64         
 13  home_form_5           22924

In [95]:
model_df.isnull().sum()

date                    0
home_team               0
away_team               0
home_score              0
away_score              0
tournament              0
city                    0
country                 0
neutral                 0
home_elo                0
away_elo                0
elo_diff                0
match_id                0
home_form_5             0
home_avg_goals_5        0
home_avg_conceded_5     0
home_avg_goal_diff_5    0
away_form_5             0
away_avg_goals_5        0
away_avg_conceded_5     0
away_avg_goal_diff_5    0
goal_difference         0
dtype: int64

In [96]:
model_df.columns.tolist()

['date',
 'home_team',
 'away_team',
 'home_score',
 'away_score',
 'tournament',
 'city',
 'country',
 'neutral',
 'home_elo',
 'away_elo',
 'elo_diff',
 'match_id',
 'home_form_5',
 'home_avg_goals_5',
 'home_avg_conceded_5',
 'home_avg_goal_diff_5',
 'away_form_5',
 'away_avg_goals_5',
 'away_avg_conceded_5',
 'away_avg_goal_diff_5',
 'goal_difference']

In [97]:
model_df.to_csv(
    "../data/processed/model_dataset.csv",
    index=False
)